# PDI — Fase 2 | Pré-processamento sincronizado com a Fase 1

**Objetivo:** executar os quatro datasets aprovados na Fase 1 de forma reproduzível, manter anotações YOLO **idênticas byte a byte**, conservar `train` / `valid` / `test` e produzir novos ZIPs processados, relatórios e comparações no Google Drive.

**Pipeline principal (candidato):** escala de cinza → filtro de mediana 3×3. **Experimentos visuais separados:** original, cinza, mediana, correção de iluminação por divisão, equalização global, CLAHE. As alternativas **não** são aplicadas indiscriminadamente ao dataset final.

> **Atenção:** esta fase **não treina** nem demonstra melhoria de precisão de um detector. Imagens em cinza podem prejudicar detectores pré-treinados em RGB. Os originais e seus rótulos não são modificados. Vergalhões continuam exploratórios (somente 3 imagens em `train`).

**Como executar:** abra no Colab pela conta institucional, ajuste **somente a célula 2** caso a pasta do Drive seja diferente, e use **Ambiente de execução → Executar tudo**. A conclusão é declarada apenas se as auditorias e as 975 imagens / 10.037 instâncias corresponderem à Fase 1.

## 1. Bibliotecas e montagem do Google Drive

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from datetime import datetime, timezone
from contextlib import contextmanager
import hashlib, io, json, math, os, shutil, tempfile, zipfile, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
print('Python / OpenCV / Pandas / NumPy / PyYAML carregados')
print('OpenCV:', cv2.__version__)

## 2. Configuração única (mesmos caminhos da Fase 1)

In [ ]:
# Pasta da Fase 1 no Drive institucional; altere apenas se for necessário.
RAIZ_PROJETO = Path('/content/drive/MyDrive/PDI_Contagem_Objetos')

VERSOES_FINAIS = {
    'arruelas/washers_hardee': 'deduplicado.zip',
    'parafusos/roboflow_screws': 'revisado.zip',
    'porcas/nutsnbolts': 'filtrado.zip',
    'vergalhoes/rebar_counting_v1': 'original.zip',
}
REFERENCIA_FASE1 = {
    'arruelas/washers_hardee': (402, 3949, {'train':280, 'valid':80, 'test':42}),
    'parafusos/roboflow_screws': (463, 3230, {'train':387, 'valid':36, 'test':40}),
    'porcas/nutsnbolts': (107, 2832, {'train':97, 'valid':5, 'test':5}),
    'vergalhoes/rebar_counting_v1': (3, 26, {'train':3}),
}
# Processamento completo dos quatro datasets. Não trocar por amostra na entrega.
DATASETS_SELECIONADOS = tuple(VERSOES_FINAIS)
JANELA_MEDIANA = 3
SEMENTE_AMOSTRAGEM = 42
AMOSTRAS_POR_DATASET = 3
EXIGIR_MANIFESTO_FASE1 = True
GERAR_ZIPS_PROCESSADOS = True

assert JANELA_MEDIANA >= 3 and JANELA_MEDIANA % 2 == 1
assert RAIZ_PROJETO.is_dir(), f'Pasta não encontrada: {RAIZ_PROJETO}. Confira acesso institucional e caminho.'
assert set(DATASETS_SELECIONADOS) == set(REFERENCIA_FASE1), 'Entrega final exige os quatro datasets.'

BASE_FASE1 = RAIZ_PROJETO / 'resultados' / 'fase1_multidataset'
BASE_FASE2 = RAIZ_PROJETO / 'resultados' / 'fase2_preprocessamento'

print('Raiz:', RAIZ_PROJETO)
print('Datasets:', *DATASETS_SELECIONADOS, sep='\n • ')

## 3. Verificação de sincronização com o manifesto da Fase 1

O notebook procura a **execução mais recente e completa** da Fase 1 e compara o hash SHA-256 de cada ZIP aprovado. Caso os arquivos tenham mudado, a execução para; não há substituição silenciosa por `original.zip`.

In [ ]:
def sha256_arquivo(caminho, bloco=1024*1024):
    h = hashlib.sha256()
    with Path(caminho).open('rb') as f:
        for dados in iter(lambda: f.read(bloco), b''):
            h.update(dados)
    return h.hexdigest()

fontes = {}
for dataset_id in DATASETS_SELECIONADOS:
    path = RAIZ_PROJETO / 'datasets' / dataset_id / VERSOES_FINAIS[dataset_id]
    if not path.is_file():
        raise FileNotFoundError(f'ZIP aprovado não encontrado: {path}')
    fontes[dataset_id] = {
        'path': path, 'sha256': sha256_arquivo(path), 'zip':path.name
    }

manifestos = sorted(BASE_FASE1.glob('*/manifesto_execucao.json'), reverse=True) if BASE_FASE1.exists() else []
manifesto_fase1 = None
caminho_manifesto = None
for candidato in manifestos:
    try:
        obj = json.loads(candidato.read_text(encoding='utf-8'))
        if not isinstance(obj, list):
            continue
        index = {r.get('dataset_id'):r for r in obj if isinstance(r,dict)}
        if not all(k in index and index[k].get('status_execucao') == 'analisado'
                   for k in DATASETS_SELECIONADOS):
            continue
        if not all(index[k].get('sha256_zip_fonte') and
                   index[k].get('versao_fonte') == Path(VERSOES_FINAIS[k]).stem
                   for k in DATASETS_SELECIONADOS):
            continue
        manifesto_fase1, caminho_manifesto = index, candidato
        break
    except (OSError, ValueError, TypeError):
        continue

if manifesto_fase1 is None:
    if EXIGIR_MANIFESTO_FASE1:
        raise FileNotFoundError(
            'Manifesto completo da Fase 1 não encontrado em resultados/fase1_multidataset/<execucao>/manifesto_execucao.json. '
            'Confirme que a Fase 1 foi executada e os resultados estão disponíveis nesta conta institucional.'
        )
    print('AVISO: manifesto não localizado; somente contagens e estrutura serão verificadas.')
else:
    for dataset_id, fonte in fontes.items():
        item = manifesto_fase1[dataset_id]
        assert item['sha256_zip_fonte'] == fonte['sha256'], (
            f'ZIP diferente do validado na Fase 1: {dataset_id}. '
            'Não processaremos versões divergentes.'
        )
        expected_images, expected_objects, expected_splits = REFERENCIA_FASE1[dataset_id]
        assert (item.get('imagens'), item.get('instancias_validas')) == (expected_images, expected_objects), (
            f'Manifesto da Fase 1 diverge da referência: {dataset_id}'
        )
        assert item.get('ocorrencias_problemas') == 0, f'Problemas de Fase 1: {dataset_id}'
        assert item.get('grupos_duplicatas_exatas') == 0, f'Duplicatas exatas de Fase 1: {dataset_id}'
    print('Sincronização SHA-256 aprovada! Manifesto:', caminho_manifesto)

display(pd.DataFrame([
    {'dataset':k, 'zip_aprovado':v['zip'], 'sha256':v['sha256']}
    for k,v in fontes.items()
]))

## 4. Funções de leitura, validação de rótulos e pré-processamento

Uma linha YOLO válida corresponde a uma instância, seja ela de detecção (5 tokens) ou segmentação (classe + pares de vértices). A rotina não reposiciona nem edita caixas e polígonos.

**Correção de iluminação revisada:** agora usa ganho limitado entre 0,85 e 1,15, intensidade parcial (0,65) e referência calculada a partir do próprio fundo estimado. A luminosidade média é compensada para evitar o escurecimento observado na versão anterior. **A correção continua sendo uma alternativa visual, não faz parte dos ZIPs gerados pelo pipeline principal.**

In [ ]:
EXTENSOES = {'.jpg','.jpeg','.png','.bmp','.webp'}
SPLITS = {'train','valid','test'}

def nome_seguro(nome):
    p = PurePosixPath(nome)
    return bool(nome) and not p.is_absolute() and '..' not in p.parts and '\\' not in nome

def eh_imagem(nome):
    p = PurePosixPath(nome)
    return 'images' in p.parts and p.suffix.lower() in EXTENSOES

def caminho_rotulo(nome_imagem):
    partes = list(PurePosixPath(nome_imagem).parts)
    assert partes.count('images') == 1, f'Pasta images ambígua: {nome_imagem}'
    partes[partes.index('images')] = 'labels'
    return str(PurePosixPath(*partes).with_suffix('.txt'))

def split_de(nome):
    partes = set(PurePosixPath(nome).parts)
    encontrados = partes & SPLITS
    assert len(encontrados) == 1, f'Split ausente ou ambíguo: {nome}'
    return next(iter(encontrados))

def parse_classes(bytes_yaml):
    cfg = yaml.safe_load(bytes_yaml.decode('utf-8-sig')) or {}
    nomes = cfg.get('names')
    if isinstance(nomes, list):
        classes = {i: str(n) for i,n in enumerate(nomes)}
    elif isinstance(nomes, dict):
        classes = {int(i): str(n) for i,n in nomes.items()}
    else:
        raise AssertionError('data.yaml sem classes names válidas')
    assert classes, 'Nenhuma classe encontrada'
    return classes

def validar_linha_yolo(linha, classes):
    tokens = linha.split()
    assert len(tokens) >= 5, 'YOLO exige classe e coordenadas'
    classe = int(tokens[0])
    assert classe in classes, f'Classe {classe} não declarada'
    cs = [float(v) for v in tokens[1:]]
    assert all(math.isfinite(v) for v in cs), 'Coordenadas não finitas'
    eps = 1e-8
    if len(cs) == 4:
        x,y,w,h = cs
        assert 0 < w <= 1 and 0 < h <= 1, 'Dimensão inválida'
        assert -eps <= x-w/2 <= 1+eps and -eps <= x+w/2 <= 1+eps, 'Caixa X fora'
        assert -eps <= y-h/2 <= 1+eps and -eps <= y+h/2 <= 1+eps, 'Caixa Y fora'
        return classe, 'deteccao'
    assert len(cs) >= 6 and len(cs) % 2 == 0, 'Polígono malformado'
    assert all(0 <= v <= 1 for v in cs), 'Vértice fora dos limites'
    assert max(cs[::2]) - min(cs[::2]) > eps, 'Polígono degenerado X'
    assert max(cs[1::2]) - min(cs[1::2]) > eps, 'Polígono degenerado Y'
    return classe, 'segmentacao'

def decodificar_imagem(dados, nome):
    im = cv2.imdecode(np.frombuffer(dados,dtype=np.uint8), cv2.IMREAD_COLOR)
    if im is None or im.ndim != 3 or im.shape[2] != 3:
        raise ValueError(f'Imagem ilegível: {nome}')
    return im

def processar_principal(bgr):
    cinza = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mediana = cv2.medianBlur(cinza, JANELA_MEDIANA)
    assert mediana.shape == bgr.shape[:2] and mediana.dtype == np.uint8
    return mediana

def corrigir_iluminacao_suave(cinza, sigma=35, intensidade=0.65,
                             ganho_min=0.85, ganho_max=1.15):
    """Corrige variações lentas de iluminação, sem escurecimento global.

    A imagem permanece em uint8 e com as mesmas dimensões. Correção
    fotométrica somente: os rótulos YOLO não precisam ser alterados.
    """
    assert cinza.ndim == 2 and cinza.dtype == np.uint8
    assert sigma > 0 and 0 <= intensidade <= 1
    assert 0 < ganho_min <= 1 <= ganho_max

    original = cinza.astype(np.float32)
    fundo = cv2.GaussianBlur(original, (0, 0), sigmaX=sigma, sigmaY=sigma)
    # Referência derivada da própria imagem, em vez do antigo fator fixo 128.
    referencia = float(np.median(fundo))
    ganho = referencia / np.maximum(fundo, 1.0)
    ganho = np.clip(ganho, ganho_min, ganho_max)
    suavizada = original * (1.0 + intensidade * (ganho - 1.0))

    # Compensação de brilho global: alterar distribuição espacial da luz,
    # mas não escurecer ou clarear a imagem inteira durante a correção.
    diferenca_media = float(np.mean(original) - np.mean(suavizada))
    suavizada = np.clip(suavizada + diferenca_media, 0, 255)
    corrigida = np.rint(suavizada).astype(np.uint8)
    return corrigida


def alternativas(bgr):
    cinza = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mediana = cv2.medianBlur(cinza, JANELA_MEDIANA)
    corrigida = corrigir_iluminacao_suave(cinza)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return {'original':cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB),
            'cinza':cinza, 'cinza_mediana_3x3':mediana,
            'correcao_iluminacao':corrigida,
            'equalizacao_global':cv2.equalizeHist(cinza),
            'clahe':clahe.apply(cinza)}

def metrica_descritiva(img):
    g = cv2.cvtColor(img,cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    return {'brilho_medio':float(np.mean(g)), 'contraste_desvio_padrao':float(np.std(g)),
            'nitidez_var_laplaciano':float(cv2.Laplacian(g,cv2.CV_64F).var())}

def codificar_mesma_extensao(img, nome):
    ext = PurePosixPath(nome).suffix.lower()
    if ext == '.jpeg': ext = '.jpg'
    if ext == '.jpg':
        ok, buf = cv2.imencode('.jpg',img,[cv2.IMWRITE_JPEG_QUALITY,95])
    elif ext == '.png':
        ok, buf = cv2.imencode('.png',img,[cv2.IMWRITE_PNG_COMPRESSION,3])
    elif ext in {'.bmp','.webp'}:
        ok, buf = cv2.imencode(ext,img)
    else:
        raise ValueError(f'Formato não suportado: {nome}')
    if not ok: raise ValueError(f'Falha ao codificar {nome}')
    return buf.tobytes()

## 5. Auditoria de entrada e planejamento da saída

Verificamos ZIPs, YAML, rótulos, splits e quantidade de objetos **antes** de gerar os derivados. Os relatórios da Fase 2 são gravados em um diretório próprio e nunca substituem os arquivos da Fase 1.

In [ ]:
def auditar_origem(dataset_id, fonte):
    with zipfile.ZipFile(fonte) as z:
        assert z.testzip() is None, f'ZIP corrompido: {fonte}'
        infos = [i for i in z.infolist() if not i.is_dir()]
        nomes_lista = [i.filename for i in infos]
        assert len(nomes_lista) == len(set(nomes_lista)), f'Caminhos duplicados no ZIP: {fonte}'
        assert all(nome_seguro(n) for n in nomes_lista), f'Caminho inseguro no ZIP: {fonte}'
        nomes = set(nomes_lista)
        imagens = sorted(n for n in nomes if eh_imagem(n))
        ymls = [n for n in nomes if PurePosixPath(n).name == 'data.yaml']
        assert len(ymls)==1, f'Esperado exatamente um data.yaml: {dataset_id}'
        classes = parse_classes(z.read(ymls[0]))
        labels_esperados = {caminho_rotulo(i) for i in imagens}
        labels_encontrados = {n for n in nomes if 'labels' in PurePosixPath(n).parts and PurePosixPath(n).suffix=='.txt'}
        assert labels_esperados == labels_encontrados, (
            f'Rótulos ausentes ou órfãos em {dataset_id}: '
            f'faltam {len(labels_esperados-labels_encontrados)}, sobram {len(labels_encontrados-labels_esperados)}'
        )
        registros=[]; por_classe=Counter(); por_formato=Counter(); por_split=Counter()
        for imagem in imagens:
            split=split_de(imagem)
            conteudo=z.read(caminho_rotulo(imagem)).decode('utf-8-sig')
            contagens = Counter()
            for n, linha in enumerate(conteudo.splitlines(),1):
                if not linha.strip(): continue
                try:
                    cid, formato=validar_linha_yolo(linha, classes)
                except Exception as exc:
                    raise AssertionError(f'{dataset_id} / {imagem} / linha {n}: {exc}') from exc
                contagens[cid]+=1; por_classe[cid]+=1; por_formato[formato]+=1
            por_split[split]+=1
            registros.append({'dataset_id':dataset_id,'imagem':imagem,'label':caminho_rotulo(imagem),
                              'split':split,'quantidade_real':sum(contagens.values()),
                              'quantidades_por_classe':json.dumps({str(k):v for k,v in sorted(contagens.items())}),
                              'sha256_label':hashlib.sha256(z.read(caminho_rotulo(imagem))).hexdigest()})
    esperado_imagens, esperado_objetos, esperado_splits=REFERENCIA_FASE1[dataset_id]
    assert (len(imagens),sum(r['quantidade_real'] for r in registros)) == (esperado_imagens, esperado_objetos), (
        f'Inventário diferente da Fase 1 para {dataset_id}'
    )
    assert dict(por_split)==esperado_splits, f'Splits divergentes da Fase 1: {dataset_id}: {dict(por_split)}'
    return {'dataset_id':dataset_id,'imagens':imagens,'registros':registros,
            'classes':classes,'por_classe':dict(por_classe),
            'por_formato':dict(por_formato),'splits':dict(por_split),
            'instancias':sum(r['quantidade_real'] for r in registros)}

inventarios={}
for dataset_id, info in fontes.items():
    inventarios[dataset_id]=auditar_origem(dataset_id, info['path'])
    inv=inventarios[dataset_id]
    print(f"OK {dataset_id}: {len(inv['imagens'])} imagens | {inv['instancias']} objetos | {inv['splits']}")

assert sum(len(d['imagens']) for d in inventarios.values())==975
assert sum(d['instancias'] for d in inventarios.values())==10037
print('\nAUDITORIA DE ENTRADA APROVADA: 975 imagens e 10.037 objetos')

## 6. Amostras visuais — comparar os métodos sem alterar os datasets

Amostragem determinística (mesma seleção em novas execuções). Métricas são apenas descritores de luminosidade/contraste/nitidez, **não** demonstram qualidade preditiva.

**Controle da correção de iluminação:** a comparação usa a função `corrigir_iluminacao_suave`, sem multiplicador fixo de brilho. A imagem corrigida deve ser comparada lado a lado com a original e com a versão em cinza. A implementação evita mudanças grandes no brilho médio, mas não demonstra melhora na detecção.

In [ ]:
AMOSTRAS = {}
for dataset_id, inv in inventarios.items():
    rng=np.random.default_rng(SEMENTE_AMOSTRAGEM)
    imagens=inv['imagens']
    ids=sorted(rng.choice(len(imagens),size=min(AMOSTRAS_POR_DATASET,len(imagens)),replace=False))
    AMOSTRAS[dataset_id]=[imagens[int(i)] for i in ids]
    print(dataset_id, '→', len(ids), 'amostras')

# Exibir um painel por dataset, sem salvar ainda imagens intermediárias.
for dataset_id, selecionadas in AMOSTRAS.items():
    with zipfile.ZipFile(fontes[dataset_id]['path']) as z:
        imagem_nome=selecionadas[0]
        bgr=decodificar_imagem(z.read(imagem_nome),imagem_nome)
        variantes=alternativas(bgr)
        media_cinza=float(np.mean(variantes['cinza']))
        media_corrigida=float(np.mean(variantes['correcao_iluminacao']))
        print(f'  Iluminação [{dataset_id}]: brilho médio cinza={media_cinza:.1f}, corrigida={media_corrigida:.1f}, diferença={media_corrigida-media_cinza:+.1f}')
        fig, axes=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
        for ax,(nome,img) in zip(axes.flat,variantes.items()):
            ax.imshow(img,cmap='gray' if img.ndim==2 else None,vmin=0 if img.ndim==2 else None,
                      vmax=255 if img.ndim==2 else None)
            ax.set_title(nome); ax.axis('off')
        fig.suptitle(f'{dataset_id} — imagem: {PurePosixPath(imagem_nome).name}',fontsize=12)
        plt.show();plt.close(fig)

## 7. Processamento integral e criação de artefatos

Os arquivos processados conservam nomes, estrutura, `data.yaml`, `train/valid/test` e rótulos exatos dos ZIPs de origem. Cada saída é testada **localmente** antes da cópia para o Drive. A pasta possui um ID exclusivo por execução.

In [ ]:
ID_EXECUCAO=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PASTA_SAIDA=BASE_FASE2 / ID_EXECUCAO
PASTA_SAIDA.mkdir(parents=True,exist_ok=False)
PASTA_LOCAL=Path(tempfile.mkdtemp(prefix='pdi_fase2_'))
print('Saída Drive:',PASTA_SAIDA)
print('Temporários locais:',PASTA_LOCAL)

registros_processamento=[]
registros_variantes=[]
resumos=[]

try:
    for dataset_id, inv in inventarios.items():
        print(f'\nPROCESSANDO {dataset_id} — {len(inv["imagens"])} imagens')
        origem=fontes[dataset_id]['path']
        pasta_destino=PASTA_SAIDA / dataset_id
        pasta_destino.mkdir(parents=True,exist_ok=True)
        nome_zip=Path(VERSOES_FINAIS[dataset_id]).stem+'_cinza_mediana3.zip'
        temporario=PASTA_LOCAL / (dataset_id.replace('/','_')+'_'+nome_zip)
        img_set=set(inv['imagens'])
        labels={r['label'] for r in inv['registros']}
        amostras_set=set(AMOSTRAS[dataset_id])
        shapes={}; hashes_entrada_labels={}
        with zipfile.ZipFile(origem,'r') as zin, zipfile.ZipFile(temporario,'w') as zout:
            for info in zin.infolist():
                if info.is_dir():
                    continue
                nome=info.filename
                conteudo=zin.read(nome)
                if nome in img_set:
                    bgr=decodificar_imagem(conteudo,nome)
                    saida=processar_principal(bgr)
                    novo_conteudo=codificar_mesma_extensao(saida,nome)
                    recuperada=cv2.imdecode(np.frombuffer(novo_conteudo,dtype=np.uint8),cv2.IMREAD_UNCHANGED)
                    assert recuperada is not None and recuperada.ndim==2 and recuperada.shape==bgr.shape[:2], f'Falha ao validar imagem codificada: {nome}'
                    shapes[nome]=tuple(bgr.shape[:2])
                    conteudo_saida=novo_conteudo
                    antes=metrica_descritiva(bgr[:,:,::-1])
                    depois=metrica_descritiva(recuperada)
                    registros_processamento.append({
                        'dataset_id':dataset_id,'imagem':nome,'split':split_de(nome),
                        'altura':bgr.shape[0],'largura':bgr.shape[1],
                        'sha256_imagem_original':hashlib.sha256(conteudo).hexdigest(),
                        'sha256_imagem_processada':hashlib.sha256(novo_conteudo).hexdigest(),
                        **{'original_'+k:v for k,v in antes.items()},
                        **{'processada_'+k:v for k,v in depois.items()}
                    })
                    if nome in amostras_set:
                        variants=alternativas(bgr)
                        for metodo,imagem_variante in variants.items():
                            m=metrica_descritiva(imagem_variante)
                            registros_variantes.append({'dataset_id':dataset_id,'imagem':nome,'metodo':metodo,**m})
                        # Exportar painel por amostra, sem interferir nas imagens do dataset.
                        fig,axs=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
                        for ax,(metodo,variante) in zip(axs.flat,variants.items()):
                            ax.imshow(variante,cmap='gray' if variante.ndim==2 else None,
                                      vmin=0 if variante.ndim==2 else None,
                                      vmax=255 if variante.ndim==2 else None)
                            ax.set_title(metodo);ax.axis('off')
                        fig.suptitle(dataset_id+' | '+PurePosixPath(nome).name)
                        destino_painel=pasta_destino / 'comparacoes'
                        destino_painel.mkdir(parents=True,exist_ok=True)
                        fig.savefig(destino_painel / (hashlib.sha256(nome.encode()).hexdigest()[:12]+'.png'),dpi=115)
                        plt.close(fig)
                else:
                    conteudo_saida=conteudo
                    if nome in labels:
                        hashes_entrada_labels[nome]=hashlib.sha256(conteudo).hexdigest()
                # Guardar os mesmos caminhos e metadados do ZIP original; só imagens mudam.
                zout.writestr(info,conteudo_saida)
        assert len(shapes)==len(img_set)
        assert len(hashes_entrada_labels)==len(labels)
        # Reabrir o artefato produzido e comprovar identidade de todos os rótulos.
        with zipfile.ZipFile(origem,'r') as zin, zipfile.ZipFile(temporario,'r') as zout:
            assert zout.testzip() is None
            assert set(zout.namelist())==set(zin.namelist())
            for r in inv['registros']:
                nome=r['imagem']; lab=r['label']
                assert zin.read(lab)==zout.read(lab), f'Rótulo foi alterado: {lab}'
                out=cv2.imdecode(np.frombuffer(zout.read(nome),dtype=np.uint8),cv2.IMREAD_UNCHANGED)
                assert out is not None and out.ndim==2 and out.shape==shapes[nome], f'Dimensão/canal alterado incorretamente: {nome}'
            for entrada in zin.namelist():
                if entrada not in img_set and not entrada.endswith('/'):
                    assert zin.read(entrada)==zout.read(entrada), f'Arquivo auxiliar alterado: {entrada}'
        if GERAR_ZIPS_PROCESSADOS:
            destino_zip=pasta_destino / nome_zip
            assert not destino_zip.exists()
            shutil.copyfile(temporario,destino_zip)
            assert sha256_arquivo(destino_zip)==sha256_arquivo(temporario), 'Cópia no Drive corrompida'
            zip_saida=str(destino_zip)
            sha_saida=sha256_arquivo(temporario)
        else:
            zip_saida=None;sha_saida=None
        resumo={
            'dataset_id':dataset_id,'versao_entrada':fontes[dataset_id]['zip'],
            'sha256_entrada':fontes[dataset_id]['sha256'],'arquivo_processado':zip_saida,
            'sha256_processado':sha_saida,'imagens_processadas':len(shapes),
            'objetos_ground_truth':inv['instancias'],'splits':inv['splits'],
            'classes':{str(k):v for k,v in inv['classes'].items()},
            'labels_identicos':True,'dimensoes_preservadas':True,
            'canal_saida':'cinza_1_canal','pipeline':'cvtColor_BGR2GRAY + medianBlur_3x3',
            'observacao':'Imagens mudam, labels e geometria permanecem. Métricas descritivas não são desempenho de modelo.'
        }
        (pasta_destino/'resumo.json').write_text(json.dumps(resumo,ensure_ascii=False,indent=2),encoding='utf-8')
        pd.DataFrame(inv['registros']).to_csv(pasta_destino/'ground_truth_referenciado.csv',index=False,encoding='utf-8-sig')
        resumos.append(resumo)
        print('OK:',len(shapes),'imagens |',inv['instancias'],'objetos | rótulos idênticos | ZIP validado')
finally:
    shutil.rmtree(PASTA_LOCAL,ignore_errors=True)

print('\nDatasets processados:',len(resumos),'de',len(DATASETS_SELECIONADOS))

## 8. Relatórios e auditoria final de aceitação

Esta célula produz os documentos finais **somente após** a aprovação das verificações. O manifesto de origem e os hashes permitem relacionar a Fase 2 à execução exata da Fase 1.

In [ ]:
assert len(resumos)==4, 'A execução não concluiu todos os datasets'
assert sum(r['imagens_processadas'] for r in resumos)==975
assert sum(r['objetos_ground_truth'] for r in resumos)==10037
assert all(r['labels_identicos'] and r['dimensoes_preservadas'] for r in resumos)
assert all(r['arquivo_processado'] for r in resumos), 'Geração de ZIPs desativada'

metricas=pd.DataFrame(registros_processamento)
experimentos=pd.DataFrame(registros_variantes)
consolidado=pd.DataFrame([{k:r[k] for k in ['dataset_id','versao_entrada','imagens_processadas',
    'objetos_ground_truth','splits','labels_identicos','dimensoes_preservadas','arquivo_processado']} for r in resumos])
assert len(metricas)==975 and len(metricas[['dataset_id','imagem']].drop_duplicates())==975
assert len(experimentos)==sum(len(v) for v in AMOSTRAS.values())*6

metricas.to_csv(PASTA_SAIDA/'metricas_descritivas.csv',index=False,encoding='utf-8-sig')
experimentos.to_csv(PASTA_SAIDA/'comparacao_metodos_amostras.csv',index=False,encoding='utf-8-sig')
consolidado.to_csv(PASTA_SAIDA/'consolidado_fase2.csv',index=False,encoding='utf-8-sig')
manifesto={
    'execucao_em_utc':datetime.now(timezone.utc).isoformat(),
    'id_execucao':ID_EXECUCAO,'fase1_manifesto':str(caminho_manifesto) if caminho_manifesto else None,
    'fase1_vinculo_sha256': bool(manifesto_fase1 is not None),
    'parametros':{'pipeline':'escala_cinza_mediana','janela_mediana':JANELA_MEDIANA,
                  'amostras_por_dataset':AMOSTRAS_POR_DATASET,'semente':SEMENTE_AMOSTRAGEM},
    'experimento_iluminacao':{'metodo':'ganho_espacial_suave','sigma':35,'intensidade':0.65,'limites_ganho':[0.85,1.15],'compensacao_brilho_medio':True,'aplicado_nos_zips':False},
    'bibliotecas':{'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__},
    'totais':{'imagens':975,'instancias':10037,'datasets':4},
    'artefatos':resumos,
    'ressalvas':['Não há medição de acurácia de detecção','Vergalhões com apenas 3 imagens train',
                 'Confirmar similaridade visual entre splits e padronizar classes antes de treinar']
}
(PASTA_SAIDA/'manifesto_fase2.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2),encoding='utf-8')

criterios=pd.DataFrame([{'dataset_id':r['dataset_id'],
                        'contagem_imagens_ok':r['imagens_processadas']==REFERENCIA_FASE1[r['dataset_id']][0],
                        'contagem_instancias_ok':r['objetos_ground_truth']==REFERENCIA_FASE1[r['dataset_id']][1],
                        'splits_ok':r['splits']==REFERENCIA_FASE1[r['dataset_id']][2],
                        'labels_identicos':r['labels_identicos'],
                        'dimensoes_preservadas':r['dimensoes_preservadas'],
                        'zip_resultante_presente':Path(r['arquivo_processado']).is_file()}
                       for r in resumos])
criterios.to_csv(PASTA_SAIDA/'criterios_aceitacao_fase2.csv',index=False,encoding='utf-8-sig')
assert criterios.drop(columns=['dataset_id']).all(axis=None), 'Critérios da Fase 2 não foram aprovados'

print('=== FASE 2 — CRITÉRIOS DE ACEITAÇÃO ===')
display(criterios)
print('\n=== CONSOLIDADO FASE 2 ===')
display(consolidado.drop(columns=['arquivo_processado']))
print('\nFASE 2: PROCESSAMENTO E SINCRONIZAÇÃO ESTRUTURAL APROVADOS')
print('Total: 975 imagens | 10.037 objetos anotados')
print('Resultados:',PASTA_SAIDA)

## 9. Interpretação, limitações e próximos passos

**O que foi verificado:** os quatro arquivos ZIP de entrada correspondem (SHA-256) à execução validada da Fase 1; o inventário e as contagens por dataset são iguais; os ZIPs derivados preservam estrutura de pastas, nomes, splits, dimensões e **todos os rótulos e arquivos auxiliares byte a byte**; as imagens de saída têm 1 canal e passaram no teste de leitura; comparações e relatórios foram exportados.

**O que ainda não foi comprovado:** superioridade da mediana sobre os outros métodos na identificação dos objetos, completude visual de cada anotação e ausência de near-duplicates entre splits. A transformação pode reduzir informações importantes para modelos treinados em imagens RGB. Para uma avaliação preditiva futura, compare o mesmo detector e os mesmos splits em entradas originais e processadas, sem escolher parâmetros com base no teste.

**Onde estão as saídas:** `PDI_Contagem_Objetos/resultados/fase2_preprocessamento/<ID_EXECUCAO>/` com um ZIP derivado por dataset, `ground_truth_referenciado.csv`, painéis de comparação, `metricas_descritivas.csv`, `comparacao_metodos_amostras.csv`, `consolidado_fase2.csv`, `criterios_aceitacao_fase2.csv` e `manifesto_fase2.json`.

**Para o GitHub:** versionar este notebook e, se apropriado, os relatórios CSV/JSON de pequeno porte. Não sobrescrever as versões finais da Fase 1.

**Atualização da correção de iluminação (experimento):** substituída a normalização de intensidade fixa `×128` por ganho espacial limitado, mistura suave e compensação de brilho médio, evitando o escurecimento global da amostra de NutsNBolts. A execução do notebook deve ser repetida para atualizar painéis e arquivos de métricas; os ZIPs do pipeline principal continuam sendo gerados com cinza + mediana 3×3.